# ASEAN PTCST-v2 — build data and run forecast development

This notebook does not use Vietnam data. It reuses the existing `asean_v1_country_runs` folder/ZIP in Drive, creates a separate V2 workspace, and leaves V1 results untouched. 2024–2025 are development evidence; do not present them as an untouched final test.

In [ ]:
# Cell 1 — Mount Drive and clone the current code
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import subprocess, sys, shutil, zipfile, pandas as pd
REPO = Path('/content/kltn')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/maiphuowng205/kltn.git', str(REPO)], check=True)
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO/'requirements-colab.txt')], check=True)

In [ ]:
# Cell 2 — Reuse ASEAN V1 data already on Drive; no re-upload is needed
DRIVE = Path('/content/drive/MyDrive')
LOCAL_COUNTRY_ROOT = Path('/content/asean_v1_country_runs')
folder = DRIVE/'kltn'/'asean_v1_country_runs'
zip_path = DRIVE/'kltn'/'asean_v1_country_runs.zip'
if folder.is_dir():
    source = folder
    if LOCAL_COUNTRY_ROOT.exists(): shutil.rmtree(LOCAL_COUNTRY_ROOT)
    shutil.copytree(source, LOCAL_COUNTRY_ROOT)
elif zip_path.is_file():
    if LOCAL_COUNTRY_ROOT.exists(): shutil.rmtree(LOCAL_COUNTRY_ROOT)
    with zipfile.ZipFile(zip_path) as z: z.extractall('/content')
    if not LOCAL_COUNTRY_ROOT.exists(): raise FileNotFoundError('ZIP must contain asean_v1_country_runs at its top level.')
else:
    raise FileNotFoundError('Cannot find MyDrive/kltn/asean_v1_country_runs or asean_v1_country_runs.zip.')
print('V1 country source:', LOCAL_COUNTRY_ROOT)

In [ ]:
# Cell 3 — Build and validate a separate V2 dataset (high-RAM full ASEAN mode)
V1_SOURCE = Path('/content/asean_v1_source')
V2_DATA = Path('/content/asean_v2')
# These are generated local workspaces only; remove a partial prior build.
for generated in (V1_SOURCE, V2_DATA):
    if generated.exists(): shutil.rmtree(generated)
subprocess.run([sys.executable, str(REPO/'scripts/assemble_asean_v1_source_from_country_runs.py'), '--country-root', str(LOCAL_COUNTRY_ROOT), '--output-root', str(V1_SOURCE), '--load-all'], check=True)
subprocess.run([sys.executable, str(REPO/'scripts/build_asean_v2_dataset.py'), '--source-root', str(V1_SOURCE), '--output-root', str(V2_DATA), '--risk-min-history', '126', '--load-all'], check=True)
subprocess.run([sys.executable, str(REPO/'scripts/validate_asean_v2_contract.py'), '--data-root', str(V2_DATA)], check=True)
print('V2 dataset ready:', V2_DATA)

In [ ]:
# Cell 4 — Train the pooled ASEAN PTCST-v2 forecast model (five seeds)
# This is development training only. It does not claim a final 2024–2025 test.
RUN_FORECAST_TRAINING = True
V2_RUN = Path('/content/asean_v2_runs/pooled_ptcst')
if RUN_FORECAST_TRAINING:
    subprocess.run([sys.executable, str(REPO/'scripts/run_asean_v2_forecasts.py'), '--data-root', str(V2_DATA), '--run-root', str(V2_RUN), '--epochs', '100', '--seeds', '7,19,31,43,59'], check=True)
else:
    print('Training skipped. Change RUN_FORECAST_TRAINING to True when ready.')

In [ ]:
# Cell 5 — Save dataset report and forecast outputs to Drive
DRIVE_OUT = DRIVE/'kltn'/'asean_v2_development'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copytree(V2_DATA/'reports', DRIVE_OUT/'dataset_reports', dirs_exist_ok=True)
if V2_RUN.exists(): shutil.copytree(V2_RUN, DRIVE_OUT/'pooled_ptcst', dirs_exist_ok=True)
print('Saved to:', DRIVE_OUT)

In [ ]:
# Cell 6 — Locked forecast metrics for all five PTCST-v2 seeds
if not RUN_FORECAST_TRAINING:
    raise RuntimeError('Run Cell 4 first so that development_predictions.npz exists.')
METRICS = Path('/content/asean_v2_forecast_metrics')
if METRICS.exists(): shutil.rmtree(METRICS)
seeds = [7, 19, 31, 43, 59]
command = [sys.executable, str(REPO/'scripts/evaluate_asean_v2_forecasts.py'), '--output-dir', str(METRICS)]
for seed in seeds:
    prediction = V2_RUN/f'seed_{seed}'/'development_predictions.npz'
    if not prediction.exists(): raise FileNotFoundError(prediction)
    command += ['--input', f'PTCST-v2_seed_{seed}={prediction}']
subprocess.run(command, check=True)
forecast_summary = pd.read_csv(METRICS/'forecast_metrics_summary.csv')
forecast_summary

In [ ]:
# Cell 7 — Seed stability and persist all forecast evaluation outputs
primary = ['mean_spearman_ic', 'median_spearman_ic', 'ic_hit_rate', 'icir', 'top_minus_bottom_5d_bps', 'dispersion_ratio', 'calibration_slope']
seed_stability = forecast_summary.groupby('country')[primary].agg(['mean', 'std']).round(6)
display(seed_stability)
EVALUATION_OUT = DRIVE_OUT/'forecast_evaluation'
if EVALUATION_OUT.exists(): shutil.rmtree(EVALUATION_OUT)
shutil.copytree(METRICS, EVALUATION_OUT)
seed_stability.to_csv(EVALUATION_OUT/'ptcst_v2_seed_stability.csv')
print('Saved forecast evaluation to:', EVALUATION_OUT)

In [ ]:
# Cell 8 — V2.1 rank-normalized ensemble, deciles and validation-only calibration
V21_OUT = V2_RUN/'v2_1_ensemble'
if V21_OUT.exists(): shutil.rmtree(V21_OUT)
subprocess.run([sys.executable, str(REPO/'scripts/run_asean_v21_ensemble.py'), '--run-root', str(V2_RUN), '--output-dir', str(V21_OUT), '--seeds', '7,19,31,43,59'], check=True)
print('V2.1 ensemble ready:', V21_OUT)

In [ ]:
# Cell 9 — Evaluate the ensemble with the same locked forecast table
ENSEMBLE_METRICS = Path('/content/asean_v21_ensemble_metrics')
if ENSEMBLE_METRICS.exists(): shutil.rmtree(ENSEMBLE_METRICS)
subprocess.run([sys.executable, str(REPO/'scripts/evaluate_asean_v2_forecasts.py'), '--output-dir', str(ENSEMBLE_METRICS), '--input', f'PTCST-v2.1-Ensemble={V21_OUT/"development_ensemble.npz"}'], check=True)
display(pd.read_csv(ENSEMBLE_METRICS/'forecast_metrics_summary.csv'))
V21_DRIVE_OUT = DRIVE_OUT/'v2_1_ensemble'
if V21_DRIVE_OUT.exists(): shutil.rmtree(V21_DRIVE_OUT)
shutil.copytree(V21_OUT, V21_DRIVE_OUT)
shutil.copytree(ENSEMBLE_METRICS, V21_DRIVE_OUT/'metrics', dirs_exist_ok=True)
print('Saved V2.1 ensemble package to:', V21_DRIVE_OUT)

In [ ]:
# Cell 10 — Select risk-aversion on validation only
LAMBDA_GRID = [2, 5, 10, 20, 50]
VALIDATION_GRID = Path('/content/asean_v21_validation_lambda_grid')
if VALIDATION_GRID.exists(): shutil.rmtree(VALIDATION_GRID)
VALIDATION_GRID.mkdir(parents=True, exist_ok=True)
lambda_rows = []
for lam in LAMBDA_GRID:
    run_dir = VALIDATION_GRID/f'lambda_{lam}'
    subprocess.run([sys.executable, str(REPO/'scripts/run_asean_v2_daily_backtest.py'), '--data-root', str(V2_DATA), '--prediction-file', str(V21_OUT/'validation_ensemble.npz'), '--run-dir', str(run_dir), '--risk-aversion', str(lam), '--cost-scenario', 'C0'], check=True)
    perf = pd.read_csv(run_dir/'portfolio_metrics_summary.csv')
    rel = pd.read_csv(run_dir/'reliability_metrics.csv')
    for row in perf.to_dict('records'):
        lambda_rows.append({'risk_aversion':lam,'country':row.get('country'),'country_sharpe':row.get('annualized_net_sharpe'),'eligible_countries':None})
    eligible = rel[(rel.covariance_fallback_rate.fillna(1) <= .10) & (rel.evaluation_coverage.fillna(0) >= .95)]
    lambda_rows.append({'risk_aversion':lam,'country':'__mean_eligible__','country_sharpe':perf.loc[perf.country.isin(eligible.country), 'annualized_net_sharpe'].mean(),'eligible_countries':len(eligible)})
lambda_detail = pd.DataFrame(lambda_rows)
lambda_table = lambda_detail.loc[lambda_detail.country.eq('__mean_eligible__')].rename(columns={'country_sharpe':'mean_country_sharpe'}).sort_values(['eligible_countries','mean_country_sharpe'], ascending=[False,False])
lambda_by_country = lambda_detail.loc[~lambda_detail.country.eq('__mean_eligible__')].copy()
display(lambda_table)
if lambda_table.empty or lambda_table.iloc[0].eligible_countries < 3: raise RuntimeError('No risk-aversion candidate passed validation reliability gates.')
SELECTED_LAMBDA = float(lambda_table.iloc[0].risk_aversion)
print('Validation-locked risk aversion:', SELECTED_LAMBDA)

In [ ]:
# Cell 11 — Run calibrated ensemble MVO under C0/C1/C2 on development
COST_RUNS = Path('/content/asean_v21_cost_scenarios')
if COST_RUNS.exists(): shutil.rmtree(COST_RUNS)
cost_rows = []
for scenario in ['C0', 'C1', 'C2']:
    run_dir = COST_RUNS/scenario
    command = [sys.executable, str(REPO/'scripts/run_asean_v2_daily_backtest.py'), '--data-root', str(V2_DATA), '--prediction-file', str(V21_OUT/'development_ensemble.npz'), '--run-dir', str(run_dir), '--risk-aversion', str(SELECTED_LAMBDA), '--cost-scenario', scenario]
    result = subprocess.run(command, text=True, capture_output=True)
    print(f'{scenario} returncode={result.returncode}')
    if result.stdout: print(result.stdout)
    if result.returncode != 0:
        if result.stderr: print(result.stderr)
        raise RuntimeError(f'{scenario} backtest failed; see stderr above.')
    perf = pd.read_csv(run_dir/'portfolio_metrics_summary.csv'); perf.insert(0, 'cost_scenario', scenario); cost_rows.append(perf)
cost_summary = pd.concat(cost_rows, ignore_index=True)
display(cost_summary)
FINAL_V21_OUT = DRIVE_OUT/'v2_1_portfolio'
if FINAL_V21_OUT.exists(): shutil.rmtree(FINAL_V21_OUT)
shutil.copytree(COST_RUNS, FINAL_V21_OUT/'cost_scenarios')
cost_summary.to_csv(FINAL_V21_OUT/'cost_sensitivity_summary.csv', index=False)
lambda_table.to_csv(FINAL_V21_OUT/'validation_lambda_selection.csv', index=False)
lambda_by_country.to_csv(FINAL_V21_OUT/'validation_lambda_by_country.csv', index=False)
print('Saved V2.1 portfolio package to:', FINAL_V21_OUT)